# L4 · Normalización y procesadores

La normalización parece un detalle técnico. En el robot fue el error que
invalidó nuestra primera evaluación de ACT: usamos la red sin su
preprocesador y los números salían sin sentido. Este notebook explica por qué
existe, las variantes y dónde vive en LeRobot.

- **Lo principal (1–4):** el problema, las cuatro formas de normalizar, el
  pipeline de procesadores y qué se rompe si falta.
- **Para profundizar (5–6):** cómo la normalización decide cuánto "pesa" cada
  articulación en el entrenamiento, y cómo cambiarla.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy.load import load_policy
from toolkit.validation.episodes import load_episodes

plt.rcParams["figure.dpi"] = 110
ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
POLICY_DIR = CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model"
CAMERA = "observation.images.cam_right_high"

stats = json.loads((ROOT / "meta" / "stats.json").read_text())
names = json.loads((ROOT / "meta" / "info.json").read_text())["features"]["action"]["names"][0]
episodes = load_episodes(ROOT)
actions = np.concatenate([e.action for e in episodes])

## 1. El problema: 26 números en escalas distintas

Los brazos se miden en radianes; las manos, de 0 (abierta) a 1 (cerrada). Y
aun entre articulaciones del brazo, unas se mueven mucho y otras casi nada.

In [ ]:
std = actions.std(axis=0)
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.bar(names, std, color=["tab:blue" if "Hand" not in n else "tab:orange" for n in names])
ax.set_ylabel("desvío de la acción"); ax.set_title("cuánto se mueve cada articulación (azul: brazos en rad · naranja: manos 0–1)")
plt.xticks(rotation=90, fontsize=7); plt.tight_layout()
print(f"la que más se mueve varía {std.max() / std.min():.0f} veces más que la que menos")

¿Por qué importa? Dos razones:

1. **Para aprender.** Las redes se entrenan mejor si todas las entradas están
   en rangos parecidos, centradas en cero. Si una entrada va de 0 a 0,05 y otra
   de −2 a 2, los pesos de la primera tienen que ser enormes para que importe,
   y el optimizador sufre.
2. **Para el error.** El entrenamiento minimiza el error de las 26 acciones
   sumado. Sin normalizar, un error de 0,05 vale lo mismo en una articulación
   que recorre 2 rad que en una que recorre 0,1 rad... aunque en la segunda es
   un error enorme. Las articulaciones chicas quedarían descuidadas.

La solución: antes de la red, llevar cada dimensión a una escala común; a la
salida, deshacerlo.

## 2. Las cuatro formas de normalizar

LeRobot trae estas (`NormalizationMode`), cada una con sus estadísticas del
dataset:

| Modo | Fórmula | Rango resultante | Sensible a extremos |
|---|---|---|---|
| `MEAN_STD` | (x − media) / desvío | sin límite, casi todo entre −3 y 3 | algo |
| `MIN_MAX` | 2·(x − mín)/(máx − mín) − 1 | exactamente [−1, 1] | **mucho**: un solo valor raro estira todo |
| `QUANTILES` | igual pero con q01 y q99 | casi todo en [−1, 1] | poco |
| `QUANTILE10` | igual pero con q10 y q90 | la mitad central en [−1, 1] | muy poco |

Nuestra ACT usa `MEAN_STD` para todo (es el default de ACT). π0.5 usa
cuantiles 1/99 (notebook 04, investigación).

Veámoslas sobre una articulación de la mano, que tiene mucho tiempo en 0
(abierta) y picos al cerrar:

In [ ]:
j = names.index("kRightHandIndex")
x = actions[:, j]
s = {k: np.array(v)[j] for k, v in stats["action"].items() if k != "count"}
modes = {
    "crudo": x,
    "MEAN_STD": (x - s["mean"]) / s["std"],
    "MIN_MAX": 2 * (x - s["min"]) / (s["max"] - s["min"]) - 1,
    "QUANTILES": 2 * (x - s["q01"]) / (s["q99"] - s["q01"]) - 1,
    "QUANTILE10": 2 * (x - s["q10"]) / (s["q90"] - s["q10"]) - 1,
}
fig, axes = plt.subplots(1, 5, figsize=(17, 2.8))
for ax, (label, values) in zip(axes, modes.items()):
    ax.hist(values, bins=60); ax.set_title(label)
    if label != "crudo":
        ax.axvline(-1, ls="--", c="0.5"); ax.axvline(1, ls="--", c="0.5")
plt.suptitle(names[j], y=1.04); plt.tight_layout()

Fijate `QUANTILE10`: como más del 10% del tiempo la mano está abierta, q10
es prácticamente 0 (0,002) y q90 es 0,53; los cierres más fuertes (hasta 0,63)
quedan pasados de 1. Y `MIN_MAX` depende de un solo valor, el máximo: un único
cierre raro cambiaría la escala de todo. Cada modo tiene su caso: no hay uno "mejor" en abstracto.

## 3. Dónde vive: el pipeline de procesadores

En LeRobot la normalización **no está dentro de la red**: está en dos
*pipelines* que se guardan junto al modelo.

In [ ]:
loaded = load_policy(POLICY_DIR, device="cpu")
print("preprocesador :", [type(step).__name__ for step in loaded.preprocessor.steps])
print("postprocesador:", [type(step).__name__ for step in loaded.postprocessor.steps])
print()
print("archivos del checkpoint:")
for path in sorted(POLICY_DIR.iterdir()):
    print("  ", path.name)

- **Rename**: cambia nombres de claves si el robot llama distinto a algo.
- **AddBatchDimension**: la red espera lotes; una observación suelta se vuelve
  un lote de 1.
- **Device**: mueve todo a CPU o GPU.
- **Normalizer**: aplica la fórmula con las estadísticas guardadas en el
  `.safetensors`.
- A la salida, **Unnormalizer** hace la inversa sobre la acción.

Las estadísticas viajan **con el modelo**, no con el dataset: así, al usar el
modelo en el robot, se normaliza exactamente como en el entrenamiento.

Recorremos el pipeline paso a paso con una observación real:

In [ ]:
dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav")
frame = dataset[3000]
observation = {k: frame[k] for k in ["observation.state", CAMERA]}

steps = ["entrada"] + [type(step).__name__ for step in loaded.preprocessor.steps]
for label, transition in zip(steps, loaded.preprocessor.step_through(observation)):
    obs = transition["observation"]
    state, image = obs["observation.state"], obs[CAMERA]
    print(f"{label:34s} estado {str(tuple(state.shape)):8s} primeros valores {np.round(state.flatten()[:3].numpy(), 3)}"
          f" · imagen media {float(image.mean()):+.3f}")

## 4. Qué se rompe si falta

Exactamente lo que nos pasó en el robot: darle a la red la observación cruda.
Comparamos, sobre varios cuadros de episodios no vistos, el error de la
predicción bien hecha contra la hecha sin procesadores.

In [ ]:
split = json.loads((REPO_ROOT / "learning" / "act" / "split.json").read_text())
policy = loaded.policy

errors = {"con procesadores": [], "sin preprocesador": [], "sin postprocesador": []}
with torch.no_grad():
    for episode in split["heldout_episode_indices"]:
        start = int(dataset.meta.episodes[episode]["dataset_from_index"])
        for offset in [30, 120, 210]:
            frame = dataset[start + offset]
            target = torch.stack([dataset[start + offset + k]["action"] for k in range(policy.config.chunk_size)])
            observation = {k: frame[k] for k in ["observation.state", CAMERA]}

            good = loaded.postprocessor(policy.predict_action_chunk(loaded.preprocessor(observation)))[0]
            raw_batch = {k: v.unsqueeze(0) for k, v in observation.items()}
            no_pre = loaded.postprocessor(policy.predict_action_chunk(raw_batch))[0]
            no_post = policy.predict_action_chunk(loaded.preprocessor(observation))[0]

            errors["con procesadores"].append((good - target).abs().mean().item())
            errors["sin preprocesador"].append((no_pre - target).abs().mean().item())
            errors["sin postprocesador"].append((no_post - target).abs().mean().item())

pd.Series({k: np.mean(v) for k, v in errors.items()}).round(3).to_frame("error medio (rad)")

- **Sin postprocesador**: la red acierta, pero en unidades normalizadas, y
  mandarías esos números como si fueran radianes. El error se multiplica por
  15: en el robot sería evidente enseguida.
- **Sin preprocesador**: la red recibe la imagen en 0–1 en vez de normalizada y
  el estado en radianes en vez de "desvíos". El error casi se duplica... y esto
  es lo peligroso: **las acciones siguen pareciendo razonables**. Es un bug que
  no se ve a simple vista y degrada todo en silencio.

Ninguno de los dos tira un error: los dos devuelven acciones de la forma
correcta. Por eso `toolkit.policy.load` siempre carga los tres juntos.

### Las imágenes

La cámara se normaliza con media y desvío de **ImageNet** (el dataset de fotos
con que se preentrenó el ResNet18), no con los de nuestro dataset. El backbone
aprendió a ver con esa escala; mantenerla le permite aprovechar lo aprendido.

In [ ]:
image_stats = {k: np.array(v).ravel() for k, v in stats[CAMERA].items() if k in ("mean", "std")}
policy_image = loaded.preprocessor.steps[-1]._tensor_stats[CAMERA]
print("guardado en el dataset:", np.round(image_stats["mean"], 3), "· desvío", np.round(image_stats["std"], 3))
print("lo que usa ACT        :", np.round(policy_image["mean"].ravel().numpy(), 3), "· desvío", np.round(policy_image["std"].ravel().numpy(), 3), "(ImageNet)")

**Ojo con el desvío que guarda el dataset: 0,009 es falso.** El desvío real
de los píxeles ronda 0,25. Es un bug de LeRobot 0.4.1: calcula las
estadísticas de imagen sobre enteros de 8 bits y el desvío de cada episodio
sale exactamente 0. Lo que queda (0,009) es sólo cuánto varía el brillo
**entre** episodios. Lo comprobamos:

In [ ]:
from lerobot.datasets.compute_stats import get_feature_stats

fake = np.random.default_rng(0).integers(0, 256, size=(20, 3, 32, 32), dtype=np.uint8)
print("desvío según LeRobot 0.4.1 (uint8):", get_feature_stats(fake, axis=(0, 2, 3), keepdims=True)["std"].ravel())
print("desvío real (numpy)              :", fake.std(axis=(0, 2, 3)).round(1))

En la versión en desarrollo de LeRobot ya está corregido. **A ACT no lo
afecta**: por defecto el entrenamiento reemplaza las estadísticas de la cámara
por las de ImageNet (`use_imagenet_stats`). Afectaría a una política entrenada
con `--dataset.use_imagenet_stats=false`: dividiría los píxeles por 0,009.
Anotado en las trampas del toolkit.

---

# Para profundizar

## 5. La normalización decide cuánto pesa cada articulación

El error de entrenamiento se calcula **en el espacio normalizado**. Con
`MEAN_STD`, un error de 1 en una articulación significa "un desvío típico de
esa articulación". Traducido a radianes, cada articulación pesa como
1 / desvío: las que se mueven poco pesan mucho.

In [ ]:
weight = 1 / np.array(stats["action"]["std"])
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.bar(names, weight / weight.mean(), color=["tab:blue" if "Hand" not in n else "tab:orange" for n in names])
ax.axhline(1, ls="--", c="0.5")
ax.set_ylabel("peso relativo en la pérdida"); ax.set_title("cuánto le importa a la pérdida 1 unidad cruda de error, por articulación")
plt.xticks(rotation=90, fontsize=7); plt.tight_layout()

Es razonable (un error de 0,05 rad importa más en una muñeca que se mueve poco)
pero tiene un riesgo: una articulación **casi congelada** tiene desvío
diminuto, y su ruido de sensor se convierte en "muchos desvíos". La red gasta
capacidad en aprender ruido. Por eso el notebook 04 busca articulaciones
congeladas, y por eso algunos labs prefieren cuantiles.

## 6. Cambiar la normalización

Se elige por tipo de dato al entrenar:

```
lerobot-train --policy.type=act \
    --policy.normalization_mapping='{"VISUAL": "MEAN_STD", "STATE": "MEAN_STD", "ACTION": "QUANTILES"}' ...
```

Dos cosas a tener en cuenta:

- **Fuga de datos**: las estadísticas salen de `meta/stats.json`, calculado
  sobre **todo** el dataset. Si el dataset incluye los episodios de evaluación,
  se cuelan en la normalización (pasó con la primera ACT, notebook 04). Al
  partir el dataset con `split_dataset` (L2), cada parte tiene sus propias
  estadísticas.
- **Cambiar la normalización es cambiar el modelo**: un checkpoint entrenado
  con un modo no se puede usar con otro.

## Referencias

- `lerobot/processor/normalize_processor.py` y `pipeline.py` (dentro de `.venv`).
- `lerobot/configs/types.py`: `NormalizationMode`.
- Para la normalización de π0.5 y TRI, `docs/08_validacion_de_datos.md`.